# Sesión 3 — Taller práctico: ¿podemos confiar en lo que vemos? (CONECTA, proyecto A: Retención)

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 3 de 9  
**Tipo:** Notebook práctico

## Instrucciones de entrega

- Equipos de 3 personas. Entreguen este notebook **ejecutado de inicio a fin** como
  `DATAVIZ_sesion_3_proyecto_A_grupo_Y.ipynb` en el plazo acordado en clase, con los archivos que el notebook deja en
  `salidas/` (la tabla limpia y las UPZ candidatas).
- Completen las celdas `# TODO` en el orden indicado (los conteos dependen del orden) y escriban en las celdas
  **Su respuesta**.
- Pueden apoyarse en un asistente de IA para concentrarse en lo importante y en las decisiones; declaren al final
  para qué lo usaron.

## Problemática de negocio

CONECTA S.A. es un operador de telecomunicaciones en Bogotá con dos líneas, **Hogar** (internet fijo) y **Móvil**
(pospago y prepago), con datos sintéticos del curso. La Gerencia de Retención lleva al comité de diciembre una
propuesta de campaña contra las cancelaciones para el primer trimestre de 2027. En la S2 el equipo calculó una tasa
por localidad y por segmento con datos sin revisar, y vio 38 formas de escribir 19 localidades y filas con pago 0.
Antes de llevar un hallazgo al comité hay que responder: **¿qué cuenta como una cancelación y en qué datos se puede
confiar?**

Al final del taller el equipo guarda en `salidas/` la **tabla limpia** y las **10 UPZ candidatas** con las que empieza la S4. La S4 parte de la tabla limpia del curso, que es idéntica a la suya si
aplicaron bien las reglas (el solucionario lo verifica).

## Preparación del entorno

In [ ]:
import pathlib

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

In [ ]:
FUENTE_CONECTA = "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026)."
crudo = pd.read_csv(DATOS / "conecta_clientes_mes.csv", dtype={"mes": str})
upz = gpd.read_file(DATOS / "upz.geojson")
print(f"{miles(len(crudo))} filas; {miles(crudo['cliente_id'].nunique())} clientes; columnas: {list(crudo.columns)}")
d = crudo.copy()   # la tabla de trabajo: cada regla de la Tarea 2 la modifica

## Tarea 1 — Perfil de calidad

Hagan un perfil de cada columna: tipo, porcentaje de nulos y número de valores distintos. Compárenlo con el
diccionario de datos (`datos/diccionario_conecta.md`) y marquen las columnas que tienen más valores distintos de los
que deberían, las que tienen nulos y las que repiten información de otra columna (por ejemplo, un nombre que se
deduce de un código).

*Consulten en la presentación:* diapositiva 10, «Cuatro categorías de calidad» (sección «La calidad de los datos»).

In [ ]:
# TODO: construyan el perfil (perfil) con tipo, % de nulos y valores distintos por columna.
# perfil = pd.DataFrame({"tipo": crudo.dtypes.astype(str),
#                        "nulos_pct": ...,
#                        "distintos": ...})
# display(perfil)
# ¿Repite una columna lo que dice otra? Por ejemplo, ¿cuántas formas de escribir la localidad hay por código de UPZ?
# crudo.groupby("codigo_upz")["localidad"].nunique().max()
# Su código aquí:

**Clasifiquen los problemas del perfil en las categorías de Wang y Strong (1996) y digan qué columnas repiten información de otra.**

*Consulten en la presentación:* diapositiva 9, «La calidad de un dato es su aptitud para el uso» (sección «La calidad de los datos»); diapositiva 10, «Cuatro categorías de calidad» (sección «La calidad de los datos»).

**Su respuesta:** *(Una viñeta por problema: columna, categoría y dimensión. Cierren con las columnas que repiten información de otra.)*

## Tarea 2 — Reglas de limpieza

Apliquen las reglas en este orden:

1. Eliminen las filas duplicadas exactas, la cuenta de prueba `CNX-PRUEBA-01` y las cuentas con pago 0 en **todos**
   los meses (empleados y pruebas).
2. Nombres de localidad y de plan: la celda ya está hecha; ejecútenla y lean qué hace.
3. Coordenadas: hay filas con latitud y longitud intercambiadas (latitud cerca de −74) y filas sin coordenadas.
   Corrijan las intercambiadas y decidan qué hacer con las nulas.

Cada celda imprime cuántos registros afecta cada regla. Al final de la tarea, redacten la bitácora.

*Consulten en la presentación:* diapositiva 31, «La bitácora completa» (sección «La bitácora»).

In [ ]:
# TODO: apliquen la regla 1 sobre d e impriman cuántos registros afecta cada parte.
# n = d.duplicated().sum(); d = d.drop_duplicates()
# print("Filas duplicadas exactas eliminadas:", n)
# La cuenta de prueba y las cuentas internas (el máximo de lo que pagan en el año es 0):
# internas = d.groupby("cliente_id")["ingreso_mensual_cop"].max()
# Su código aquí:

In [ ]:
# Ejecute esta celda sin modificarla: nombre oficial de la localidad según el código de UPZ y plan unificado.
import unicodedata
oficial = upz.set_index("codigo_upz")["localidad"]
n = (d["localidad"] != d["codigo_upz"].map(oficial)).sum()
d["localidad"] = d["codigo_upz"].map(oficial)
print("Filas con la localidad escrita distinto del nombre oficial:", miles(n))
PLANES = {"basico": "Básico", "estandar": "Estándar", "plus": "Plus"}
sin_tildes = lambda s: "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
plan = d["plan"].map(lambda p: PLANES[sin_tildes(p).strip().lower()])
n = (d["plan"] != plan).sum(); d["plan"] = plan
print("Filas con el plan escrito distinto:", miles(n))
print(d["localidad"].nunique(), "localidades;", sorted(d["plan"].unique()))

In [ ]:
# TODO: corrijan las coordenadas intercambiadas e impriman cuántas filas tienen coordenadas cambiadas y nulas.
# cambiadas = d["latitud"] < -60
# Su código aquí:

**Bitácora de la Tarea 2.** Una viñeta por regla: problema, decisión, por qué y cuántos registros afecta (la cifra que imprimió su código).

**Su respuesta:** *(Una viñeta por regla.)*

## Tarea 3 — ¿Qué cuenta como cancelación?

`cancelo` vale 1 también cuando el cliente **cambió de plan** (`tipo_cancelacion == "Cambio de plan"`): el cliente
sigue con CONECTA.

1. Creen la columna `perdida` (cancelación sin cambio de plan) e impriman cuántas filas cambian de definición.
2. En una figura de dos paneles, comparen `cancelo` y `perdida` **por estrato** y **por segmento** (el segmento lo
   definieron en la S2).
3. Con las pérdidas, la tabla de tipos por segmento (`tipos`, en % de las pérdidas de cada segmento): muestra cómo
   se va el cliente que se va (voluntaria, por mora o sin recarga).
4. Comparen el **ranking de UPZ** con las dos definiciones: solo UPZ con al menos 80 clientes (con menos, la tasa es
   inestable, como el intervalo de la Candelaria en la S2; criterio del curso).

*Consulten en la presentación:* diapositiva 6, «La regla, cliente por cliente»; diapositiva 36, «Los segmentos de clientes» (sección «La bitácora»).

In [ ]:
# TODO: creen perdida, grafiquen cancelo frente a perdida por estrato y por segmento, y calculen tipos.
# d["perdida"] = ((d["cancelo"] == 1) & (d["tipo_cancelacion"] != "Cambio de plan")).astype(int)
# print("Filas con cambio de plan:", ...)
# d["segmento"] = np.where(d["linea"] == "Hogar", "Hogar", np.where(d["modalidad"] == "Prepago", "Móvil prepago", "Móvil pospago"))
# por_estrato = 100 * d.groupby("estrato")[["cancelo", "perdida"]].mean()
# por_segmento = 100 * d.groupby("segmento")[["cancelo", "perdida"]].mean()
# fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [2, 1]})
# barras agrupadas: cancelo en gris, perdida en verde
# tipos = 100 * pd.crosstab(d.loc[d["perdida"] == 1, "segmento"], d.loc[d["perdida"] == 1, "tipo_cancelacion"], normalize="index")
# Su código aquí:

In [ ]:
# TODO: construyan el ranking de UPZ (al menos 80 clientes) con cancelo y con perdida, y comparen los puestos.
# ranking = d.groupby(["codigo_upz", "upz", "localidad"]).agg(clientes=("cliente_id", "nunique"),
#                                                           cancelo=("cancelo", "mean"), perdida=("perdida", "mean"))
# ranking = ranking[ranking["clientes"] >= 80].copy()
# ranking["puesto_cancelo"] = ranking["cancelo"].rank(ascending=False, method="first").astype(int)
# ranking["puesto_perdida"] = ...
# k = round(0.10 * len(ranking))          # el 10 % peor
# ¿Qué UPZ entran al 10 % peor con una definición y no con la otra? ¿Qué pasa con La Esmeralda?
# Su código aquí:

**Bitácora de la Tarea 3.** Una viñeta por regla: problema, decisión, por qué y cuántos registros afecta (la cifra que imprimió su código).

**Su respuesta:** *(Una viñeta por regla.)*

**¿Qué cambia al contar solo las pérdidas: por estrato, por segmento y en el ranking de UPZ? ¿Por qué importa para la campaña?**

*Consulten en la presentación:* diapositiva 6, «La regla, cliente por cliente»; diapositiva 36, «Los segmentos de clientes» (sección «La bitácora»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 4 — Datos faltantes: el NPS

El NPS falta en cerca del 79 % de las filas. Calculen el porcentaje de respuesta según las quejas del mes (0, 1, 2,
3 o más). Si faltara completamente al azar (MCAR), la respuesta sería pareja (van Buuren, 2018, §1.2). Registren en
la bitácora su decisión sobre el NPS.

In [ ]:
# TODO: calculen respuesta_nps por número de quejas y grafíquenla.
# respuesta_nps = d.assign(responde=d["nps"].notna(), quejas=d["quejas_mes"].clip(upper=3))
# respuesta_nps = 100 * respuesta_nps.groupby("quejas")["responde"].mean()
# Su código aquí:

**¿El NPS falta al azar? ¿Qué deciden hacer con él y para qué puede servir todavía?**

*Consulten en la presentación:* diapositiva 23, «Datos faltantes: ¿de qué depende que un dato falte?» (sección «Los problemas del caso»); diapositiva 25, «Cómo saber cuál es» (sección «Los problemas del caso»); diapositiva 27, «QUÉ HACER CON LOS FALTANTES» (sección «Los problemas del caso»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**Bitácora de la Tarea 4.** Una viñeta por regla: problema, decisión, por qué y cuántos registros afecta (la cifra que imprimió su código).

**Su respuesta:** *(Una viñeta por regla.)*

## Tarea 5 — Tabla limpia y UPZ candidatas

1. Guarden `d` como `salidas/conecta_limpio_s3.csv`.
2. Las candidatas: la tasa de pérdida por UPZ (`tasa_pct`, en %) y el porcentaje de filas prepago (`pct_prepago`, en
   %), solo para las UPZ con al menos 80 clientes. Son el **10 % peor**: las `k = round(0,10 × n)` UPZ con la tasa más
   alta. Guárdenlas en `salidas/upz_candidatas_s3.csv`: son la entrada de la S4.

In [ ]:
# TODO: guarden la tabla limpia en SALIDAS.
# d = d.drop(columns="segmento").reset_index(drop=True)     # el segmento se calcula; no se guarda
# d.to_csv(SALIDAS / "conecta_limpio_s3.csv", index=False, encoding="utf-8")
# Su código aquí:

In [ ]:
# TODO: calculen por_upz, elijan las k UPZ candidatas y guárdenlas en SALIDAS.
# por_upz = d.groupby(["codigo_upz", "upz", "localidad"]).agg(clientes=("cliente_id", "nunique"),
#                                                         tasa_pct=("perdida", lambda s: 100 * s.mean()),
#                                                         pct_prepago=("modalidad", lambda s: 100 * (s == "Prepago").mean()))
# por_upz = por_upz[por_upz["clientes"] >= 80]
# k = round(0.10 * len(por_upz))
# candidatas = por_upz.nlargest(k, "tasa_pct")
# candidatas.reset_index().to_csv(SALIDAS / "upz_candidatas_s3.csv", index=False, encoding="utf-8")
# Su código aquí:

**¿Qué UPZ llevarían a la S4 como candidatas y qué deberían verificar antes de llevarlas al comité?**

*Consulten en la presentación:* diapositiva 36, «Los segmentos de clientes» (sección «La bitácora»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Preguntas de interpretación

**1. ¿Qué regla de limpieza cambió más la conclusión del proyecto y por qué?**

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Interpretación final para el negocio

Para la Gerencia de Retención, en 5–8 viñetas: qué problemas tenían los datos; qué se corrigió y qué no; cuánto
cambió la tasa frente a la de la S2; qué cambió del párrafo «Lo que llevamos a la S4» de la S2; qué UPZ pasan a la S4;
uso de IA.

**Su respuesta:** *(Escriba aquí sus viñetas.)*